In [150]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [151]:
# Make dataframe
path = r"C:\Users\13926\Atlantic-Project\florida_real_estate_clean.csv"
df = pd.read_csv(path)
df.head()

,type,lastSoldPrice,sqft,stories,beds,baths,baths_full,garage,age,lat,long
0,single_family,605000,2274.0,1.0,2.0,3.0,2.0,2.0,19.0,26.4517,-80.1580
1,single_family,285000,2170.0,1.0,3.0,2.0,2.0,2.0,46.0,27.4287,-81.3519
2,condos,425000,1722.0,1.0,3.0,2.0,2.0,2.0,10.0,26.5228,-81.7065
3,single_family,596000,1699.0,1.0,3.0,3.0,3.0,0.0,74.0,25.9850,-80.1407
4,single_family,165000,640.0,1.0,1.0,1.0,1.0,0.0,55.0,29.2219,-81.0095


In [152]:
df.shape

(10870, 11)

In [153]:
df['type'].value_counts()

type
single_family                  7096
condos                         1701
townhomes                       756
land                            631
mobile                          560
multi_family                    110
condo_townhome_rowhome_coop      13
farm                              3
Name: count, dtype: int64

In [154]:
df_encoded = pd.get_dummies(
    df,
    columns=['type'],
    drop_first=True,
)

In [155]:
df_encoded.dtypes

lastSoldPrice           int64
sqft                  float64
stories               float64
beds                  float64
baths                 float64
baths_full            float64
garage                float64
age                   float64
lat                   float64
long                  float64
type_condos              bool
type_farm                bool
type_land                bool
type_mobile              bool
type_multi_family        bool
type_single_family       bool
type_townhomes           bool
dtype: object

In [156]:
# Prepare the data for machine learning
X = df_encoded.drop(columns=['lastSoldPrice'])
y = df_encoded['lastSoldPrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [157]:
X.baths_full.value_counts()

baths_full
2.0     7640
3.0     1828
1.0      858
4.0      385
5.0       92
6.0       36
7.0       16
8.0        9
9.0        4
20.0       1
11.0       1
Name: count, dtype: int64

In [158]:
# Set IQR bounds
def get_iqr_bounds(series, multiplier=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return lower, upper

num_cols = X.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    lower, upper = get_iqr_bounds(X_train[col])
    n_outliers = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    print(f'{col}: bounds=({lower:.2f}, {upper:.2f}), outliers={n_outliers} ({n_outliers/len(X_train)*100:.1f}%)')


sqft: bounds=(171.12, 3280.12), outliers=471 (5.4%)
stories: bounds=(1.00, 1.00), outliers=2002 (23.0%)
beds: bounds=(-1.00, 7.00), outliers=16 (0.2%)
baths: bounds=(0.50, 4.50), outliers=309 (3.6%)
baths_full: bounds=(2.00, 2.00), outliers=2589 (29.8%)
garage: bounds=(-3.00, 5.00), outliers=17 (0.2%)
age: bounds=(-48.50, 99.50), outliers=52 (0.6%)
lat: bounds=(23.32, 32.32), outliers=0 (0.0%)
long: bounds=(-84.35, -79.21), outliers=661 (7.6%)


In [160]:
# Cap outliers
cols_to_check = ['sqft', 'beds', 'baths', 'garage', 'age']
def cap_outliers(X, columns, multiplier=1.5):
    X = X.copy()
    for col in columns:
        lower, upper = get_iqr_bounds(X[col], multiplier)
        X[col] = X[col].clip(lower, upper)
    return X

X_train_capped = cap_outliers(X_train, cols_to_check)
X_test_capped = cap_outliers(X_test, cols_to_check)

In [161]:
# Scale the features to prevent featrues with large numbers from unfairly dominating calculations
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_capped)
X_test_scaled = scaler.transform(X_test_capped)

In [162]:
# Find the best number of neighbors for the KNN regression model using cross-validation
param_grid = {'n_neighbors': range(1, 31, 2)}
grid = GridSearchCV(KNeighborsRegressor(), param_grid, cv=5, scoring='neg_mean_absolute_error')
grid.fit(X_train_scaled, y_train)
print(grid.best_params_)


{'n_neighbors': 17}


In [163]:
print(grid.best_score_)

-212004.24845797848


In [164]:
# Fit model
knn_regressor = KNeighborsRegressor(n_neighbors=grid.best_params_['n_neighbors'], weights='uniform')
knn_regressor.fit(X_train_scaled, y_train)
y_train_pred = knn_regressor.predict(X_train_scaled)
y_pred = knn_regressor.predict(X_test_scaled)
print(f'MSE (training): {mean_squared_error(y_train, y_train_pred)}')
print(f'MSE (testing): {mean_squared_error(y_test, y_pred)}')
print(f'R-squared score (training): {r2_score(y_train, y_train_pred)}')
print(f'R-squared score (testing) : {r2_score(y_test, y_pred)}')



MSE (training): 833713186349.6843
MSE (testing): 375664785845.94666
R-squared score (training): 0.4294878167260584
R-squared score (testing) : 0.4730836434678918
